# Test the three Easy-to-Read strategy classifiers

This notebook loads and tests:

1. **XLM-R Large focal classifier**
2. **Multilingual-E5 Large BCE classifier**
3. **Qwen2.5-7B pairwise QLoRA classifier**

Edit the standard sentence, the Easy-to-Read rewrite, and `MODEL_CHOICE`, then run the cells in order.

The Qwen model requires a GPU and is evaluated one candidate strategy at a time using the released taxonomy cards and label-specific thresholds.


In [ ]:
# Install the required libraries.
# In Google Colab, run this once at the beginning.

%pip install -q -U transformers accelerate peft bitsandbytes huggingface_hub pandas pyyaml safetensors


## Label codes and fixed output order

The six labels always use this order:

| Numeric ID | Code | Label |
|---:|---|---|
| 0 | `SYN` | Synonymy |
| 1 | `MOD` | Modulation |
| 2 | `COMP` | Compression |
| 3 | `EXPL` | Explanation |
| 4 | `SYNT` | Syntactic Change |
| 5 | `OMIT` | Omission |


In [ ]:
# Fixed label order used by all three released models.

LABELS = [
    "Synonymy",
    "Modulation",
    "Compression",
    "Explanation",
    "Syntactic Change",
    "Omission",
]

LABEL_CODES = {
    "Synonymy": "SYN",
    "Modulation": "MOD",
    "Compression": "COMP",
    "Explanation": "EXPL",
    "Syntactic Change": "SYNT",
    "Omission": "OMIT",
}

LABEL_TO_ID = {
    label: index
    for index, label in enumerate(LABELS)
}

ID_TO_LABEL = {
    index: label
    for index, label in enumerate(LABELS)
}

MODEL_REPOSITORIES = {
    "xlmr": "hannah-khallaf/e2r-strategy-xlmr-large-focal",
    "e5": "hannah-khallaf/e2r-strategy-multilingual-e5-large-bce",
    "qwen": "hannah-khallaf/e2r-strategy-qwen2.5-7b-pairwise-qlora",
}

FALLBACK_THRESHOLDS = {
    "xlmr": {label: 0.46 for label in LABELS},
    "e5": {label: 0.23 for label in LABELS},
    "qwen": {
        "Synonymy": 0.63,
        "Modulation": 0.54,
        "Compression": 0.09,
        "Explanation": 0.40,
        "Syntactic Change": 0.20,
        "Omission": 0.32,
    },
}

print("Label codes:")
for index, label in enumerate(LABELS):
    print(f"{index}: {LABEL_CODES[label]:<4} -> {label}")


## Choose a model and enter a sentence pair

Set `MODEL_CHOICE` to one of:

- `"xlmr"`
- `"e5"`
- `"qwen"`


In [ ]:
# Change this value to test another model.
MODEL_CHOICE = "xlmr"

# Enter the standard sentence first.
STANDARD_SENTENCE = (
    "The committee postponed the implementation of the measure."
)

# Enter the corresponding Easy-to-Read rewrite second.
EASY_TO_READ_SENTENCE = (
    "The committee decided to use the measure later."
)

assert MODEL_CHOICE in MODEL_REPOSITORIES, (
    "MODEL_CHOICE must be 'xlmr', 'e5', or 'qwen'."
)

print("Selected model:", MODEL_CHOICE)
print("Standard sentence:", STANDARD_SENTENCE)
print("Easy-to-Read sentence:", EASY_TO_READ_SENTENCE)


## Shared utilities

In [ ]:
from __future__ import annotations

import gc
import importlib
import inspect
import json
import sys
from pathlib import Path
from typing import Any

import pandas as pd
import torch
import yaml
from huggingface_hub import hf_hub_download
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)


def select_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def clear_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def thresholds_from_model(
    model: Any,
    model_choice: str,
) -> dict[str, float]:
    config_data = getattr(
        model.config,
        "e2r_classifier",
        None,
    )

    if isinstance(config_data, dict):
        saved = config_data.get("thresholds")
        if isinstance(saved, dict):
            return {
                label: float(saved[label])
                for label in LABELS
            }

    return FALLBACK_THRESHOLDS[model_choice].copy()


def make_result_table(
    probabilities: dict[str, float],
    thresholds: dict[str, float],
) -> pd.DataFrame:
    rows = []

    for label in LABELS:
        probability = float(probabilities[label])
        threshold = float(thresholds[label])

        rows.append(
            {
                "id": LABEL_TO_ID[label],
                "code": LABEL_CODES[label],
                "label": label,
                "probability": probability,
                "threshold": threshold,
                "predicted": probability >= threshold,
            }
        )

    return pd.DataFrame(rows)


def print_prediction_summary(results: pd.DataFrame) -> None:
    predicted = results.loc[
        results["predicted"],
        ["code", "label"],
    ]

    if predicted.empty:
        print("Predicted labels: none")
        return

    formatted = [
        f"{row.code} ({row.label})"
        for row in predicted.itertuples()
    ]

    print("Predicted labels:", ", ".join(formatted))


## XLM-R and Multilingual-E5 inference

Both encoder models return six logits in the fixed label order. The notebook applies `sigmoid` and the threshold saved with the model.

For Multilingual-E5, the required prefix `query: ` is added to **both** sentences.


In [ ]:
def load_encoder_model(model_choice: str):
    if model_choice not in {"xlmr", "e5"}:
        raise ValueError(
            "Encoder choice must be 'xlmr' or 'e5'."
        )

    repo_id = MODEL_REPOSITORIES[model_choice]
    device = select_device()

    tokenizer = AutoTokenizer.from_pretrained(
        repo_id,
        trust_remote_code=True,
    )

    load_kwargs = {
        "trust_remote_code": True,
    }

    if device.type == "cuda":
        load_kwargs["torch_dtype"] = torch.float16

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            repo_id,
            **load_kwargs,
        )
    )

    model.to(device)
    model.eval()

    return model, tokenizer, device


def predict_with_encoder(
    model_choice: str,
    standard_sentence: str,
    easy_to_read_sentence: str,
) -> pd.DataFrame:
    model, tokenizer, device = load_encoder_model(
        model_choice
    )

    if model_choice == "e5":
        first_sequence = "query: " + standard_sentence
        second_sequence = "query: " + easy_to_read_sentence
    else:
        first_sequence = standard_sentence
        second_sequence = easy_to_read_sentence

    inputs = tokenizer(
        first_sequence,
        second_sequence,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        logits = model(**inputs).logits[0]
        values = torch.sigmoid(logits).float().cpu()

    probabilities = {
        label: float(values[index])
        for index, label in enumerate(LABELS)
    }

    thresholds = thresholds_from_model(
        model,
        model_choice,
    )

    results = make_result_table(
        probabilities,
        thresholds,
    )

    del model
    del tokenizer
    clear_memory()

    return results


## Qwen2.5 pairwise QLoRA inference

The Qwen classifier tests each candidate label separately. For each label it:

1. inserts the official taxonomy card;
2. asks whether that strategy is present;
3. computes the probability of the next token being a `true` rather than a `false` variant;
4. applies the released label-specific threshold.

A CUDA GPU is required for the 4-bit Qwen model.


In [ ]:
def download_qwen_reference_package(
    repo_id: str,
    local_root: str = "./qwen_e2r_reference",
) -> Path:
    root = Path(local_root).resolve()

    filenames = [
        "reference_implementation/__init__.py",
        "reference_implementation/binary_relevance.py",
        "reference_implementation/taxonomy.py",
        "e2r_taxonomy.yaml",
    ]

    for filename in filenames:
        hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir=root,
        )

    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    importlib.invalidate_caches()
    return root


def construct_taxonomy(
    taxonomy_module: Any,
    taxonomy_path: Path,
) -> Any:
    taxonomy_class = taxonomy_module.E2RTaxonomy
    taxonomy_data = yaml.safe_load(
        taxonomy_path.read_text(encoding="utf-8")
    )

    attempts = []

    for name in (
        "load_taxonomy",
        "load_e2r_taxonomy",
        "read_taxonomy",
    ):
        loader = getattr(taxonomy_module, name, None)
        if callable(loader):
            attempts.extend(
                [
                    lambda loader=loader: loader(taxonomy_path),
                    lambda loader=loader: loader(str(taxonomy_path)),
                ]
            )

    for name in dir(taxonomy_class):
        lowered = name.lower()

        if not any(
            token in lowered
            for token in ("yaml", "file", "path", "load")
        ):
            continue

        loader = getattr(taxonomy_class, name, None)

        if callable(loader):
            attempts.extend(
                [
                    lambda loader=loader: loader(taxonomy_path),
                    lambda loader=loader: loader(str(taxonomy_path)),
                ]
            )

    attempts.extend(
        [
            lambda: taxonomy_class(taxonomy_path),
            lambda: taxonomy_class(str(taxonomy_path)),
            lambda: taxonomy_class(taxonomy_data),
        ]
    )

    errors = []

    for attempt in attempts:
        try:
            taxonomy = attempt()
            taxonomy.render_macro_card(
                LABELS[0],
                include_descendants=True,
                include_examples=True,
            )
            return taxonomy
        except Exception as error:
            errors.append(
                f"{type(error).__name__}: {error}"
            )

    signature = inspect.signature(taxonomy_class)

    raise RuntimeError(
        "Could not construct E2RTaxonomy. "
        f"Constructor signature: {signature}. "
        "Last errors: "
        + " | ".join(errors[-5:])
    )


def load_qwen_model_and_helpers():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Qwen requires a CUDA GPU. "
            "In Colab, select Runtime > Change runtime type > GPU."
        )

    from peft import AutoPeftModelForCausalLM

    repo_id = MODEL_REPOSITORIES["qwen"]

    reference_root = download_qwen_reference_package(
        repo_id
    )

    from reference_implementation import (
        binary_relevance,
        taxonomy as taxonomy_module,
    )

    taxonomy_path = (
        reference_root / "e2r_taxonomy.yaml"
    )

    taxonomy = construct_taxonomy(
        taxonomy_module,
        taxonomy_path,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        repo_id,
        trust_remote_code=False,
    )

    quantisation = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoPeftModelForCausalLM.from_pretrained(
        repo_id,
        quantization_config=quantisation,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=False,
    )

    model.eval()

    return (
        model,
        tokenizer,
        taxonomy,
        binary_relevance,
    )


def render_qwen_prompt(
    tokenizer: Any,
    system_prompt: str,
    user_prompt: str,
) -> str:
    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def predict_with_qwen(
    standard_sentence: str,
    easy_to_read_sentence: str,
) -> pd.DataFrame:
    (
        model,
        tokenizer,
        taxonomy,
        binary_relevance,
    ) = load_qwen_model_and_helpers()

    row = {
        "source_text": standard_sentence,
        "simplified_text": easy_to_read_sentence,
    }

    probabilities = {}

    for label in LABELS:
        system_prompt, user_prompt = (
            binary_relevance.build_binary_prompt(
                row,
                label,
                taxonomy,
                include_definition=True,
                demonstrations=(),
                request_confidence=False,
                response_format="boolean",
                input_mode="pair",
            )
        )

        prompt = render_qwen_prompt(
            tokenizer,
            system_prompt,
            user_prompt,
        )

        probability = (
            binary_relevance.boolean_token_probability(
                model,
                tokenizer,
                prompt,
            )
        )

        probabilities[label] = float(probability)

        print(
            f"Finished {LABEL_CODES[label]} "
            f"({label}): {probability:.6f}"
        )

    thresholds = FALLBACK_THRESHOLDS["qwen"]

    results = make_result_table(
        probabilities,
        thresholds,
    )

    del model
    del tokenizer
    clear_memory()

    return results


## Run the selected model

Change `MODEL_CHOICE` in the earlier input cell and rerun this cell to test another model.


In [ ]:
if MODEL_CHOICE in {"xlmr", "e5"}:
    results = predict_with_encoder(
        MODEL_CHOICE,
        STANDARD_SENTENCE,
        EASY_TO_READ_SENTENCE,
    )
else:
    results = predict_with_qwen(
        STANDARD_SENTENCE,
        EASY_TO_READ_SENTENCE,
    )

print()
print_prediction_summary(results)

display(
    results.style.format(
        {
            "probability": "{:.4f}",
            "threshold": "{:.2f}",
        }
    )
)


## Optional: return only the predicted label codes

In [ ]:
predicted_codes = results.loc[
    results["predicted"],
    "code",
].tolist()

predicted_labels = results.loc[
    results["predicted"],
    "label",
].tolist()

output = {
    "model": MODEL_CHOICE,
    "standard_sentence": STANDARD_SENTENCE,
    "easy_to_read_sentence": EASY_TO_READ_SENTENCE,
    "predicted_codes": predicted_codes,
    "predicted_labels": predicted_labels,
    "scores": {
        row.label: float(row.probability)
        for row in results.itertuples()
    },
}

print(json.dumps(
    output,
    indent=2,
    ensure_ascii=False,
))
